<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/plan/tc2-hasta-s10-20261105/Cuadernos/Taller_Control_2.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir TC2 en Google Colab"/></a>

# Taller de Control 2 · Del buscador al explorador de evidencias

**Universidad Central · Maestría en Analítica de Datos · Big Data 2026-2S**

**Para una pareja o una persona sola. 100 puntos.** El segundo taller evalúa competencias **nuevas** de S07, S09 y S10, no repite el TC1.

**El caso de Laura.** Ya tiene expedientes de SECOP organizados. Ahora una consulta sobre compras públicas falla cuando los redactores usan palabras distintas. Necesita entender el ranking, recuperar documentos por significado y explorar imágenes y voces sin atribuirles contenidos ni relaciones falsas.

**Tu producto:** un *Explorador de Evidencias* HTML generado desde tus búsquedas, con un gráfico propio y diez grabaciones humanas licenciadas.

**Antes de comenzar:** Archivo → Guardar una copia en Drive. Conecta CPU en Colab; no hace falta GPU ni instalación local. Una pareja usa una sola copia. No pegues claves ni compartas los enlaces privados de Atlas.

**Tutorial paralelo:** [Checklist TC2, paso a paso](https://github.com/jazaineam1/BigData2026/blob/plan/tc2-hasta-s10-20261105/assets/tutoriales/tc2-paso-a-paso.html). El checklist es ayuda, no evaluación.

## Mapa de la experiencia · del problema al producto

| Desafío | Nueva pregunta | Producto propio | Puntos |
|---|---|---|---:|
| Inicio | ¿Por qué dos frases equivalentes devuelven resultados distintos? | Predicción guardada | Formativo |
| E1 · BM25 | ¿Por qué un texto ocupa el primer lugar? | Dos Top-10 y un experimento con una palabra | 20 |
| E2 · Semántica | ¿Qué documentos responden sin usar las mismas palabras? | Vectores E5, coseno y prueba vectorial | 25 |
| E3 · Evidencia | ¿Cuál buscador recupera información pertinente? | Precision@5, Recall@5, RRF y elección | 15 |
| E4 · Imagen | ¿Qué información conserva una imagen reducida? | PNG de tu Top-5, histograma y dos perfiles | 20 |
| E5 · Audio | ¿Qué se puede medir de una voz sin transcribirla? | Diez audios, RMS y FFT | 15 |
| E6 · Explorador | ¿Cómo presentar resultados honestos en un solo lugar? | Página HTML y defensa estructurada | 5 |
| **Total** | | | **100** |

**Cada bloque:** predice → experimenta → observa → corrige → decide. **No te califican por pulsar Ejecutar todo.** El puntaje nace de resultados y decisiones propios; el validador reabre archivos. Los *scores* de BM25, coseno y RRF no son directamente comparables entre sí.

## Rúbrica completa · 18 controles, 100 puntos

| Control | Lo que debe comprobarse | Puntos |
|---|---|---:|
| E1.1 | Reconocer tokens y analizador léxico | 6 |
| E1.2 | Construir dos rankings BM25 correctos | 8 |
| E1.3 | Cambiar una palabra y recalcular | 6 |
| E2.1 | Generar embeddings E5, N × 384, prefijos apropiados | 7 |
| E2.2 | Calcular rankings Top-5 por coseno | 8 |
| E2.3 | Demostrar búsqueda Atlas real o ruta local identificada y documentada | 6 |
| E2.4 | Cambiar Top-k y comprobar diferencias | 4 |
| E3.1 | Precision@5 y Recall@5 con juicios independientes | 6 |
| E3.2 | Combinar rankings usando RRF | 5 |
| E3.3 | Elegir, descartar alternativa y reconocer límite | 4 |
| E4.1 | Leer RGB, luminancia e histograma del propio gráfico | 6 |
| E4.2 | Transformar imagen en dos perfiles y medir diferencias | 8 |
| E4.3 | Elegir perfil por legibilidad observable | 6 |
| E5.1 | Obtener diez voces reales, distintas y licenciadas | 5 |
| E5.2 | Medir RMS, FFT y resolución en dos ventanas | 6 |
| E5.3 | Distinguir propiedades de la señal de transcripción | 4 |
| E6.1 | Explorador offline con dos fuentes separadas | 3 |
| E6.2 | Defensa integrada basada en resultados | 2 |
| **Total** | | **100** |

**Nota orientativa TC1:** con entrega, 1 + 4 × puntos/100, un decimal al registrar; sin entrega, 0. **El PDA del repositorio es anterior al curso 2026:** la ponderación oficial del taller debe confirmar el docente. Las comprobaciones automáticas no validan por sí solas la autenticidad del uso remoto de Atlas.

## Preparación: identidad, bibliotecas y datos

**No repetiremos TC1.** Te entregamos un corpus ya preparado de **200 procesos SECOP** y cinco preguntas profesionales etiquetadas para evaluar relevancia. Los objetos contractuales provienen del repositorio del curso; los juicios de pertinencia son para esta muestra de enseñanza, no para todo SECOP.

**Qué necesitas:** cuenta Google, Colab CPU y conexión para descargar el modelo E5 y los diez audios (CC BY 4.0). Puedes recuperar avance en Drive. No necesitas usar Cassandra, Neo4j ni descargar SECOP por páginas.

**Acciones:** EJECUTA corre una celda; ELIGE cambia un parámetro; COMPLETA exige modificar una variable; COMPRUEBA relee archivos; REVISAR muestra tus resultados. Nunca hay respuestas académicas en prosa libre.

In [ ]:
#@title Preparar · Instalar bibliotecas y cargar motor versionado (no evaluado) { display-mode: "form" }
import os,sys,subprocess,importlib.util,json,urllib.request
from pathlib import Path
paquetes={"numpy":"numpy>=1.26","pandas":"pandas>=2.0","pyarrow":"pyarrow>=17","PIL":"pillow>=10","soundfile":"soundfile>=0.12","sentence_transformers":"sentence-transformers>=3,<6","pymongo":"pymongo>=4.8","matplotlib":"matplotlib>=3.8"}
faltan=[spec for mod,spec in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltan:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*faltan])
CORE_SOURCE = "# -*- coding: utf-8 -*-\n\"\"\"TC2 · motor de ejercicios, evaluador por archivos y exportación (Big Data UC).\nLa autoevaluación técnica NO sustituye la revisión del docente. Sin credenciales.\n\"\"\"\nfrom __future__ import annotations\nimport base64, collections, hashlib, html, io, json, math, re, shutil, unicodedata, wave\nfrom pathlib import Path\nimport numpy as np\n\nSCHEMA = \"tc2-v1\"\nMODELO = \"intfloat/multilingual-e5-small\"\nRUBRICA = [\n (\"E1.1\",\"BM25: tokens y mecanismo\",6),(\"E1.2\",\"BM25: rankings literal y paráfrasis\",8),\n (\"E1.3\",\"BM25: modificar una palabra y comprobar\",6),\n (\"E2.1\",\"E5: vectores 384 y prefijos\",7),(\"E2.2\",\"Similitud coseno y top 5\",8),\n (\"E2.3\",\"Atlas vectorial real o contrato local\",6),(\"E2.4\",\"Experimento con candidatos o filtro\",4),\n (\"E3.1\",\"Precision y Recall con juicios externos\",6),(\"E3.2\",\"RRF calculado por posiciones\",5),\n (\"E3.3\",\"Elección con alternativa y límite\",4),\n (\"E4.1\",\"RGB, luminancia e histograma\",6),(\"E4.2\",\"Dos transformaciones medibles\",8),\n (\"E4.3\",\"Perfil que conserva legibilidad\",6),\n (\"E5.1\",\"Diez voces humanas licenciadas\",5),(\"E5.2\",\"RMS y FFT en dos ventanas\",6),\n (\"E5.3\",\"Interpretación acústica y límite\",4),\n (\"E6.1\",\"Explorador de dos orígenes\",3),(\"E6.2\",\"Decisiones ancladas al resultado\",2)]\nassert len(RUBRICA)==18 and sum(r[2] for r in RUBRICA)==100\nURL_VOCES = \"https://huggingface.co/datasets/PolyAI/minds14/resolve/main/es-ES/train-00000-of-00001.parquet\"\nSHA_VOCES = \"989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf\"\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\"rb\") as f:\n        for b in iter(lambda:f.read(1024*1024),b\"\"): h.update(b)\n    return h.hexdigest()\n\ndef escribir(path, data):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False)+\"\\n\",encoding=\"utf-8\")\n    return path\n\ndef leer(path):\n    return json.loads(Path(path).read_text(encoding=\"utf-8\"))\n\ndef normalizar(s):\n    return \"\".join(c for c in unicodedata.normalize(\"NFKD\",str(s).lower()) if not unicodedata.combining(c))\n\ndef tokens(s):\n    return re.findall(r\"[a-z0-9]{2,}\",normalizar(s))\n\ndef texto(doc):\n    return (doc[\"nombre_proceso\"]+\" \"+doc[\"descripcion\"]).strip()\n\ndef corpus_lista(base):\n    data=leer(Path(base)/\"tc2_corpus_2026.json\")\n    docs=data[\"documentos\"]\n    ids=[d[\"id_proceso\"] for d in docs]\n    if not docs or len(ids)!=len(set(ids)):raise ValueError(\"Corpus duplicado o vacío\")\n    return docs\n\ndef intenciones(base):\n    data=leer(Path(base)/\"tc2_juicios_2026.json\")\n    return data[\"intenciones\"]\n\ndef elegir_intencion(base,alias):\n    opciones=intenciones(base)\n    pos=int(hashlib.sha256(str(alias).strip().encode()).hexdigest(),16)%len(opciones)\n    return opciones[pos]\n\ndef bm25(docs,consulta,k=10):\n    \"\"\"BM25 local sobre tokens Unicode normalizados; no es Elasticsearch.\"\"\"\n    q=tokens(consulta);toks=[tokens(texto(d)) for d in docs]\n    cnt=[collections.Counter(t) for t in toks];n=len(cnt)\n    avdl=sum(map(len,toks))/max(n,1)\n    df=collections.Counter(w for t in cnt for w in t)\n    scores=[]\n    for d,c,t in zip(docs,cnt,toks):\n        s=0.\n        for w in q:\n            if not c.get(w):continue\n            idf=math.log(1+(n-df[w]+.5)/(df[w]+.5))\n            freq=c[w];s+=idf*(freq*2.2)/(freq+1.2*(.25+.75*len(t)/max(avdl,1)))\n        scores.append({\"id_proceso\":d[\"id_proceso\"],\"score\":round(s,8)})\n    return sorted(scores,key=lambda x:(-x[\"score\"],x[\"id_proceso\"]))[:k]\n\ndef coseno(docs,m,q,k=10):\n    m=np.asarray(m,dtype=np.float32);q=np.asarray(q,dtype=np.float32)\n    if m.shape!=(len(docs),384) or q.shape!=(384,):raise ValueError(\"E5 debe generar N×384 y query vector 384\")\n    if not np.isfinite(m).all() or not np.isfinite(q).all():raise ValueError(\"Vector no finito\")\n    norma=np.linalg.norm(m,axis=1)*np.linalg.norm(q)\n    if np.min(norma)<=1e-9:raise ValueError(\"Vector nulo\")\n    sc=(m@q)/norma\n    return sorted([{\"id_proceso\":d[\"id_proceso\"],\"score\":round(float(s),7)} for d,s in zip(docs,sc)],key=lambda x:(-x[\"score\"],x[\"id_proceso\"]))[:k]\n\ndef medidas(ranking,relevantes,k=5):\n    ids=[r[\"id_proceso\"] for r in ranking[:k]]\n    rel=set(relevantes);aciertos=len(set(ids)&rel)\n    return {\"aciertos\":aciertos,\"precision_at_5\":round(aciertos/k,4),\"recall_at_5\":round(aciertos/max(len(rel),1),4),\"relevantes_totales\":len(rel)}\n\ndef fusion_rrf(a,b,k0=60,top=10):\n    puntos=collections.defaultdict(float)\n    for ranked in (a,b):\n        for pos,r in enumerate(ranked,1):puntos[r[\"id_proceso\"]]+=1/(k0+pos)\n    return [{\"id_proceso\":i,\"score\":round(v,10)} for i,v in sorted(puntos.items(),key=lambda x:(-x[1],x[0]))[:top]]\n\ndef csv_image(ranking,docs,path):\n    \"\"\"PNG académico derivado del Top-5 verdadero; no implica fotos de contratos.\"\"\"\n    from PIL import Image,ImageDraw,ImageFont\n    lookup={d[\"id_proceso\"]:d for d in docs};rs=ranking[:5]\n    im=Image.new(\"RGB\",(960,520),\"#f5f9ff\");d=ImageDraw.Draw(im)\n    d.text((36,24),\"Top 5 del buscador — imagen generada en el TC2\",fill=\"#172c50\")\n    maxi=max((float(x[\"score\"]) for x in rs),default=1)\n    for n,r in enumerate(rs):\n        y=90+n*76;title=lookup[r[\"id_proceso\"]][\"nombre_proceso\"][:67]\n        d.text((40,y),str(n+1)+\". \"+title,fill=\"#14253f\")\n        w=int(820*float(r[\"score\"])/max(maxi,1e-9))\n        d.rectangle((40,y+24,40+max(w,0),y+41),fill=\"#1d608f\")\n        d.text((870,y+23),f'{r[\"score\"]:.3f}',fill=\"#152c4f\")\n    d.text((35,485),\"Imagen derivada de resultados SECOP; no es fotografia ni prueba de irregularidad\",fill=\"#263d57\")\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);im.save(path)\n    return path\n\ndef inspeccion_imagen(path):\n    from PIL import Image\n    im=Image.open(path).convert(\"RGB\");a=np.asarray(im)\n    y=(.2126*a[:,:,0]+.7152*a[:,:,1]+.0722*a[:,:,2])\n    hist,bins=np.histogram(y,bins=16,range=(0,256))\n    pixel=a[min(im.height-1,120),min(im.width-1,50)].tolist()\n    return {\"ancho\":im.width,\"alto\":im.height,\"pixel_rgb\":pixel,\n      \"pixel_luminancia\":round(.2126*pixel[0]+.7152*pixel[1]+.0722*pixel[2],3),\n      \"histograma\":hist.tolist(),\"suma_histograma\":int(hist.sum()),\"luminancia_media\":round(float(y.mean()),3)}\n\ndef perfil_imagen(original,dest,lado,contraste=1.0):\n    from PIL import Image,ImageEnhance\n    if not (80<=int(lado)<=960 and .5<=float(contraste)<=2.0):raise ValueError(\"Perfil fuera de rango\")\n    im=Image.open(original).convert(\"RGB\");im.thumbnail((int(lado),int(lado)))\n    im=ImageEnhance.Contrast(im).enhance(float(contraste))\n    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True);im.save(dest)\n    return {\"lado\":int(lado),\"contraste\":float(contraste),\"ancho\":im.width,\"alto\":im.height,\"bytes\":dest.stat().st_size,\"sha256\":sha(dest)}\n\ndef audio_pcm(path):\n    import soundfile as sf\n    x,sr=sf.read(str(path),dtype=\"float32\",always_2d=False)\n    if x.ndim==2:x=x.mean(axis=1)\n    if sr<=0 or len(x)==0:raise ValueError(\"Audio PCM vacío\")\n    return x,int(sr)\n\ndef metricas_voz(path,ventana=512):\n    x,sr=audio_pcm(path);n=int(ventana)\n    if n not in (256,512,1024,2048):raise ValueError(\"Ventana fuera del laboratorio\")\n    rms=float(np.sqrt(np.mean(np.square(x.astype(np.float64)))))\n    zcr=float(np.mean((x[:-1]*x[1:])<0)) if len(x)>1 else 0.\n    segmento=np.pad(x[:n],(0,max(0,n-len(x))))[:n]\n    amp=np.abs(np.fft.rfft(segmento*np.hanning(n)))\n    frec=np.fft.rfftfreq(n,1/sr)\n    pico=float(frec[int(np.argmax(amp[1:])+1)]) if len(amp)>1 else 0.\n    centr=float(np.sum(frec*amp)/max(np.sum(amp),1e-12))\n    return {\"duracion_s\":round(len(x)/sr,4),\"sr\":sr,\"rms\":round(rms,7),\n       \"zcr\":round(zcr,7),\"frecuencia_pico_hz\":round(pico,3),\n       \"centroide_hz\":round(centr,3),\"resolucion_hz\":round(sr/n,3),\"ventana_fft\":n}\n\ndef obtener_voces(out,source=None):\n    \"\"\"Carga bytes REALES del MINDS14 versión fijada; falla cerrado, sin audio sintético.\"\"\"\n    import pandas as pd\n    import soundfile as sf\n    import urllib.request\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    fuente=Path(source) if source else out/\"minds14_es_ES.parquet\"\n    if not fuente.exists():\n        tmp=fuente.with_suffix(\".parquet.part\")\n        try:\n            req=urllib.request.Request(URL_VOCES,headers={\"User-Agent\":\"BigData-UCentral-TC2/1.0\"})\n            with urllib.request.urlopen(req,timeout=120) as r,open(tmp,\"wb\") as w:\n                while True:\n                    b=r.read(1024*1024)\n                    if not b:break\n                    w.write(b)\n                    if w.tell()>100_000_000:raise ValueError(\"Descarga >100 MB\")\n            tmp.replace(fuente)\n        except Exception:\n            tmp.unlink(missing_ok=True);raise\n    if sha(fuente)!=SHA_VOCES:raise ValueError(\"La fuente real no coincide con el SHA autorizado\")\n    df=pd.read_parquet(fuente,columns=[\"audio\",\"path\",\"transcription\",\"intent_class\"])\n    vistos=set();sel=[]\n    for _,r in df.sort_values(\"path\",kind=\"stable\").iterrows():\n        c=int(r[\"intent_class\"])\n        if c not in vistos:sel.append(r);vistos.add(c)\n        if len(sel)==10:break\n    if len(sel)!=10:raise ValueError(\"No hay diez intenciones distintas en el corpus real\")\n    audios=[]; hashes=set()\n    for i,r in enumerate(sel,1):\n        raw=r[\"audio\"].get(\"bytes\")\n        if not isinstance(raw,(bytes,bytearray)):raise ValueError(\"Se esperan WAV reales embebidos\")\n        original=out/f\"voz_{i:02d}_fuente.wav\";original.write_bytes(raw)\n        x,sr=sf.read(io.BytesIO(raw),dtype=\"float32\",always_2d=False)\n        if x.ndim==2:x=x.mean(axis=1)\n        pcm=out/f\"voz_{i:02d}.wav\"\n        sf.write(pcm,x,sr,subtype=\"PCM_16\",format=\"WAV\")\n        digest=sha(original)\n        if digest in hashes:raise ValueError(\"Voz duplicada\")\n        hashes.add(digest)\n        audios.append({\"n\":i,\"archivo\":str(pcm.name),\"original\":original.name,\n           \"fuente_url\":URL_VOCES,\"source_record_id\":str(r[\"path\"]),\n           \"source_original_sha256\":digest,\"sha256\":sha(pcm),\n           \"license\":\"CC BY 4.0\",\"source_kind\":\"MINDS14_es_ES\",\"source_dataset_sha256\":SHA_VOCES,\n           \"intent_class\":int(r[\"intent_class\"]),\n           \"transcripcion_referencia\":str(r[\"transcription\"]),\n           \"medidas\":metricas_voz(pcm)})\n    return audios\n\ndef hacer_html(carpeta,docs,bm,sem,rrf,datoimg,audios,decisiones):\n    \"\"\"Explorador offline: filtra resultados YA calculados; no recalcula embeddings en navegador.\"\"\"\n    carpeta=Path(carpeta);lookup={d[\"id_proceso\"]:d for d in docs}\n    def tabla(nombre,filas):\n        tr=[]\n        for x in filas[:5]:\n            d=lookup[x[\"id_proceso\"]]\n            tr.append(\"<tr><td>\"+html.escape(x[\"id_proceso\"])+\"</td><td>\"+html.escape(d[\"nombre_proceso\"][:120])+\n              \"</td><td>\"+html.escape(str(round(x[\"score\"],5)))+\n              \"</td><td><a href='\"+html.escape(d[\"url_secop\"],quote=True)+\"' target='_blank' rel='noopener'>SECOP</a></td></tr>\")\n        return \"<section><h3>\"+html.escape(nombre)+\"</h3><table><thead><tr><th>ID</th><th>Objeto</th><th>Score</th><th>Fuente</th></tr></thead><tbody>\"+\"\".join(tr)+\"</tbody></table></section>\"\n    voces=\"\".join(\"<article><strong>Voz \"+str(a[\"n\"])+\"</strong><p>PolyAI MINDS14 · CC BY 4.0 · \"+html.escape(a[\"source_record_id\"])+\n      \"</p><audio controls preload='none' src='E5/audio/\"+html.escape(a[\"archivo\"],quote=True)+\"'></audio><p>RMS \"+str(a[\"medidas\"][\"rms\"])+\n      \" · duración \"+str(a[\"medidas\"][\"duracion_s\"])+\" s · centroide \"+str(a[\"medidas\"][\"centroide_hz\"])+\" Hz</p></article>\" for a in audios)\n    style=\"\"\"body{font:16px/1.5 system-ui;margin:0;background:#f3f7fc;color:#182c44}main{max-width:1150px;margin:auto;padding:20px}header{background:#132f56;color:white;padding:25px}section,article{background:white;border:1px solid #d9e2ed;border-radius:12px;padding:16px;margin:14px 0;overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:14px}th,td{border-bottom:1px solid #dbe2ea;padding:8px;text-align:left}a{color:#145c9e}img{max-width:100%;height:auto}audio{width:100%;max-width:400px}p{max-width:90ch}@media(prefers-color-scheme:dark){body{background:#101b2a;color:#e8f0fa}section,article{background:#1c2e45;border-color:#405574}a{color:#a3caff}th,td{border-color:#3e5066}}\"\"\"\n    page=\"<!doctype html><html lang='es'><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>TC2 · Explorador de Laura</title><style>\"+style+\"</style><header><h1>Explorador de Evidencias · TC2</h1><p>SECOP: búsqueda sobre texto contractual · MINDS14: audios financieros distintos, sin vínculo documental con SECOP</p></header><main><h2>Resultados de búsqueda</h2>\"+tabla(\"BM25 léxico\",bm)+tabla(\"E5 semántico\",sem)+tabla(\"RRF híbrido\",rrf)+\"<h2>Imagen derivada del ranking</h2><section><img alt='Visualización del Top 5 generado en el taller, no fotografía de contratos' src='E4/ranking.png'><p>El histograma y la reducción representan píxeles; no describen el contenido semántico de un documento.</p><img alt='Versión de tamaño reducido' src='E4/perfil_160.png'><img alt='Versión legible' src='E4/perfil_640.png'></section><h2>Diez voces reales autorizadas</h2>\"+voces+\"<section><h2>Decisiones y límites</h2><pre>\"+html.escape(json.dumps(decisiones,ensure_ascii=False,indent=2))+\"</pre><p>Los audios no pertenecen a las noticias ni a los contratos SECOP. Una FFT no transcribe palabras.</p></section></main></html>\"\n    ruta=carpeta/\"explorador.html\";ruta.write_text(page,encoding=\"utf-8\")\n    return ruta\n\ndef empaquetar(carpeta,manifest):\n    \"\"\"Archivo para Drive, evitando cachés, secretos y fuentes externas pesadas.\"\"\"\n    import zipfile\n    carpeta=Path(carpeta)\n    no_ext={\".env\",\".pem\",\".key\"}\n    rutas=[p for p in carpeta.rglob(\"*\") if p.is_file() and not p.name.endswith(\".part\") and p.suffix.lower() not in no_ext and not p.name.startswith(\"minds14_\")]\n    prohibido=re.compile(r\"(mongodb\\+srv://|sk-proj-|BEGIN PRIVATE KEY|claimUrl)\",re.I)\n    zip_path=carpeta.parent/(carpeta.name+\".zip\")\n    with zipfile.ZipFile(zip_path,\"w\",compression=zipfile.ZIP_DEFLATED) as z:\n        for p in rutas:\n            if p.stat().st_size>50_000_000:raise ValueError(\"Un archivo de evidencia supera 50 MB\")\n            if p.suffix.lower() in {\".json\",\".jsonl\",\".html\",\".txt\"} and prohibido.search(p.read_text(encoding=\"utf-8\",errors=\"ignore\")):\n                raise ValueError(\"Se detectó un posible secreto en \"+str(p))\n            z.write(p,p.relative_to(carpeta))\n    return zip_path\n\ndef validar(carpeta,docs,intencion):\n    \"\"\"Recalcula controles desde archivos, no confía en el puntaje del manifest.\"\"\"\n    carpeta=Path(carpeta);checks={}\n    def lee(r):return leer(carpeta/r)\n    def ok(item,cond,detalle):\n        puntos=next(p for c,_,p in RUBRICA if c==item)\n        checks[item]={\"maximo\":puntos,\"puntos\":puntos if bool(cond) else 0,\"ok\":bool(cond),\"detalle\":str(detalle)[:260]}\n    def near(a,b,tol=1e-4):\n        try:return abs(float(a)-float(b))<=tol\n        except (TypeError,ValueError):return False\n    try:\n        bm=lee(\"E1/bm25.json\");q=intencion\n        expected1=bm25(docs,q[\"literal\"],10); expected2=bm25(docs,q[\"parafrasis\"],10)\n        ok(\"E1.1\",bm.get(\"tokens_literal\")==tokens(q[\"literal\"]) and bm.get(\"analizador\")==\"normalizado_simple\",\"Tokens observados y analizador.\")\n        ok(\"E1.2\",bm.get(\"literal\")==expected1 and bm.get(\"parafrasis\")==expected2,\"Dos rankings BM25 local verificables.\")\n        cambio=bm.get(\"cambio\",{});original=q[\"literal\"];nuevo=cambio.get(\"consulta\",\"\")\n        ok(\"E1.3\",nuevo!=original and len(tokens(nuevo))>0 and cambio.get(\"ranking\")==bm25(docs,nuevo,10) and bool(cambio.get(\"prediccion\")),\"Consulta modificada y ranking correcto.\")\n    except Exception as e:\n        for item in (\"E1.1\",\"E1.2\",\"E1.3\"):ok(item,False,str(e))\n        bm={}\n    try:\n        e=lee(\"E2/semantica.json\")\n        x=np.load(carpeta/\"E2/embeddings.npy\",allow_pickle=False);q1=np.load(carpeta/\"E2/query_literal.npy\",allow_pickle=False);q2=np.load(carpeta/\"E2/query_parafrasis.npy\",allow_pickle=False)\n        shape_ok=x.shape==(len(docs),384) and q1.shape==(384,) and q2.shape==(384,) and np.isfinite(x).all() and np.isfinite(q1).all()\n        ok(\"E2.1\",shape_ok and e.get(\"modelo\")==MODELO and e.get(\"prefijos\")==[\"passage:\",\"query:\"],\"N×384, modelo y prefijos.\")\n        want1=coseno(docs,x,q1,10);want2=coseno(docs,x,q2,10)\n        ok(\"E2.2\",e.get(\"literal\")==want1 and e.get(\"parafrasis\")==want2,\"Top-k coseno recomputado.\")\n        a=e.get(\"atlas\",{})\n        remoto=bool(a.get(\"ping\") and a.get(\"indice_queryable\") and a.get(\"query_ejecutada\") and len(a.get(\"resultados\",[]))>0 and a.get(\"motor\")==\"MongoDB Atlas Vector Search\")\n        contrato=bool(a.get(\"indice_path\")==\"embedding\" and a.get(\"indice_dimensiones\")==384 and a.get(\"pipeline_limit\")==5 and a.get(\"sistema\") in (\"Atlas real\",\"Ruta local declarada\"))\n        ok(\"E2.3\",contrato and remoto,\"Atlas declarado real = \"+str(remoto)+\"; validación remota definitiva pendiente de auditoría docente.\")\n        if contrato and a.get(\"sistema\")==\"Ruta local declarada\" and not remoto:\n            checks[\"E2.3\"][\"puntos\"]=3\n            checks[\"E2.3\"][\"ok\"]=False\n            checks[\"E2.3\"][\"detalle\"]=\"3/6 por contrato vectorial local, no por ejecución remota de Atlas.\"\n        varia=e.get(\"experimento\",{}); k=varia.get(\"k\")\n        ok(\"E2.4\",k in (3,8) and varia.get(\"ranking\")==coseno(docs,x,q2,int(k)) if k in (3,8) else False,\"Top-k alterado, mismo vector.\")\n    except Exception as err:\n        for item in (\"E2.1\",\"E2.2\",\"E2.3\",\"E2.4\"):ok(item,False,str(err))\n        e={}\n    try:\n        r=lee(\"E3/evaluacion.json\")\n        ranking_lex=bm.get(\"parafrasis\",[])\n        ranking_sem=e.get(\"parafrasis\",[])\n        fusion=fusion_rrf(ranking_lex,ranking_sem,60,10)\n        a=medidas(ranking_lex,intencion[\"relevantes\"]);b=medidas(ranking_sem,intencion[\"relevantes\"])\n        ok(\"E3.1\",r.get(\"bm25\")==a and r.get(\"e5\")==b,\"Precision@5 y Recall@5 sobre juicios independientes.\")\n        ok(\"E3.2\",r.get(\"rrf\")==fusion and r.get(\"metricas_rrf\")==medidas(fusion,intencion[\"relevantes\"]),\"RRF calculado desde posiciones.\")\n        de=r.get(\"decision\",{})\n        ok(\"E3.3\",de.get(\"elegido\") in (\"BM25\",\"E5\",\"RRF\") and de.get(\"descartado\") in (\"BM25\",\"E5\",\"RRF\") and de.get(\"descartado\")!=de.get(\"elegido\") and de.get(\"limite\") in (\"Solo cubre los juicios etiquetados\",\"No demuestra irregularidades\") and de.get(\"precision_observada\")=={\"BM25\":a,\"E5\":b,\"RRF\":medidas(fusion,intencion[\"relevantes\"])}[de.get(\"elegido\", \"BM25\")][\"precision_at_5\"],\"Elegir, descartar y usar métrica real.\")\n    except Exception as err:\n        for item in (\"E3.1\",\"E3.2\",\"E3.3\"):ok(item,False,str(err))\n    try:\n        meta=lee(\"E4/imagen.json\")\n        orig=carpeta/\"E4/ranking.png\"\n        m=inspeccion_imagen(orig)\n        ok(\"E4.1\",meta.get(\"inspeccion\")==m and m[\"suma_histograma\"]==m[\"ancho\"]*m[\"alto\"],\"RGB, luminancia e histograma.\")\n        files=all((carpeta/f\"E4/perfil_{n}.png\").exists() for n in (160,640))\n        both=meta.get(\"perfiles\",{})\n        a=inspeccion_imagen(carpeta/\"E4/perfil_160.png\");b=inspeccion_imagen(carpeta/\"E4/perfil_640.png\")\n        ok(\"E4.2\",files and a[\"ancho\"]<b[\"ancho\"] and both.get(\"160\",{}).get(\"sha256\")==sha(carpeta/\"E4/perfil_160.png\") and both.get(\"640\",{}).get(\"sha256\")==sha(carpeta/\"E4/perfil_640.png\"),\"Perfiles generados con diferencia observable.\")\n        ok(\"E4.3\",meta.get(\"preferido\")==\"640\" and meta.get(\"limite\")==\"Reducir puede volver ilegibles las etiquetas\" and b[\"ancho\"]>=480,\"Perfil útil por legibilidad.\")\n    except Exception as err:\n        for item in (\"E4.1\",\"E4.2\",\"E4.3\"):ok(item,False,str(err))\n    try:\n        a=lee(\"E5/audios.json\"); voces=a[\"audios\"]\n        bueno=len(voces)==10 and len({v.get(\"source_record_id\") for v in voces})==10 and len({v.get(\"intent_class\") for v in voces})==10\n        for v in voces:\n            p=carpeta/\"E5/audio\"/v[\"archivo\"]\n            original=carpeta/\"E5/audio\"/v[\"original\"]\n            bueno=bueno and p.exists() and original.exists() and sha(p)==v[\"sha256\"] and sha(original)==v.get(\"source_original_sha256\") and v.get(\"license\")==\"CC BY 4.0\" and v.get(\"source_kind\")==\"MINDS14_es_ES\" and v.get(\"source_dataset_sha256\")==SHA_VOCES\n        ok(\"E5.1\",bueno,\"Diez voces humanas distintas, procedencia y huellas.\")\n        a1=a[\"comparacion\"];v=voces[int(a1[\"posicion\"])-1];ruta=carpeta/\"E5/audio\"/v[\"archivo\"]\n        m512=metricas_voz(ruta,512);m1024=metricas_voz(ruta,1024)\n        ok(\"E5.2\",a1.get(\"ventana_512\")==m512 and a1.get(\"ventana_1024\")==m1024 and m512[\"resolucion_hz\"]>m1024[\"resolucion_hz\"],\"RMS, pico y resolución espectral.\")\n        ok(\"E5.3\",a1.get(\"limite\")==\"RMS y FFT no transcriben palabras\" and a1.get(\"referencia_no_asr\") is True and 1<=int(a1.get(\"posicion\",0))<=10,\"Distinción entre señal y transcripción.\")\n    except Exception as err:\n        for item in (\"E5.1\",\"E5.2\",\"E5.3\"):ok(item,False,str(err))\n    try:\n        report=leer(carpeta/\"E6/defensa.json\")\n        page=(carpeta/\"explorador.html\").read_text(encoding=\"utf-8\")\n        ok(\"E6.1\",\"BM25 léxico\" in page and \"E5 semántico\" in page and \"Diez voces reales autorizadas\" in page and \"sin vínculo\" in page and \"<audio\" in page,\"Explorador HTML con orígenes separados.\")\n        ok(\"E6.2\",report.get(\"modelo\") in (\"BM25\",\"E5\",\"RRF\") and report.get(\"audio_no_implica\")==\"Transcripción ni intención\" and report.get(\"imagen_no_implica\")==\"Contenido semántico del contrato\",\"Defensa de los tres mecanismos.\")\n    except Exception as err:\n        for item in (\"E6.1\",\"E6.2\"):ok(item,False,str(err))\n    total=sum(v[\"puntos\"] for v in checks.values())\n    return {\"schema\":SCHEMA,\"puntaje\":total,\"nota_provisional\":math.floor((1+4*total/100)*10+0.5+1e-9)/10,\n            \"controles\":checks,\"pendiente_revision_docente\":\"E2.3: confirmar Atlas con auditoría o aplicar contingencia documentada\"}\n"
Path("tc2_core.py").write_text(CORE_SOURCE,encoding="utf-8")
sys.modules.pop("tc2_core",None)
import tc2_core as core
print("TC2 · Motor autónomo listo. Rúbrica:",len(core.RUBRICA),"controles /",sum(r[2] for r in core.RUBRICA),"puntos.")

In [ ]:
#@title Inicio · ELIGE identidad de la pareja y si guardas avance en Drive { display-mode: "form" }
PAREJA_ID = "" #@param {type:"string"}
INTEGRANTE_1 = "" #@param {type:"string"}
CODIGO_1 = "" #@param {type:"string"}
APELLIDO_1 = "" #@param {type:"string"}
INTEGRANTE_2 = "" #@param {type:"string"}
CODIGO_2 = "" #@param {type:"string"}
APELLIDO_2 = "" #@param {type:"string"}
GUARDAR_AVANCE_EN_DRIVE = False #@param {type:"boolean"}
import re,shutil,hashlib,datetime
PAREJA= re.sub("[^a-z0-9_-]+","-",core.normalizar(PAREJA_ID.strip()))[:35].strip("-")
if not PAREJA: raise ValueError("Escribe un nombre para tu pareja (sin correos ni contraseñas).")
if not INTEGRANTE_1.strip() or not CODIGO_1.strip() or not APELLIDO_1.strip():
    raise ValueError("Completa identidad del primer integrante; los datos solo van a la entrega restringida.")
BASE=Path.cwd()/"tc2_datos";BASE.mkdir(parents=True,exist_ok=True)
TRABAJO=Path.cwd()/("TC2_"+PAREJA);TRABAJO.mkdir(parents=True,exist_ok=True)
CARPETA_DRIVE=None
if GUARDAR_AVANCE_EN_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CARPETA_DRIVE=Path("/content/drive/MyDrive/BigData2026")/("TC2_"+PAREJA)/"avance"
        if CARPETA_DRIVE.exists():
            shutil.copytree(CARPETA_DRIVE,TRABAJO,dirs_exist_ok=True)
            print("Avance anterior restaurado.")
    except Exception as error:
        print("Drive no disponible. Trabaja localmente y descarga el ZIP:",type(error).__name__)
        CARPETA_DRIVE=None
def checkpoint(nombre):
    print("CHECKPOINT",nombre,":",len(list(TRABAJO.rglob("*"))),"elementos en carpeta.")
    if CARPETA_DRIVE:
        CARPETA_DRIVE.mkdir(parents=True,exist_ok=True)
        shutil.copytree(TRABAJO,CARPETA_DRIVE,dirs_exist_ok=True)
        print("Avance protegido en Drive.")
def leer_oficial(nombre):
    destinos=[
      "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/"+nombre,
      "https://raw.githubusercontent.com/jazaineam1/BigData2026/plan/tc2-hasta-s10-20261105/Datos/"+nombre]
    for url in destinos:
        try:
            with urllib.request.urlopen(url,timeout=30) as r: b=r.read()
            obj=json.loads(b.decode("utf-8"))
            (BASE/nombre).write_bytes(b)
            return obj
        except Exception as error:
            ultimo=error
    if (BASE/nombre).exists(): return json.loads((BASE/nombre).read_text())
    raise RuntimeError("No se obtuvo "+nombre+"; comprueba tu conexión: "+str(ultimo))
_ = leer_oficial("tc2_corpus_2026.json")
_ = leer_oficial("tc2_juicios_2026.json")
DOCUMENTOS=core.corpus_lista(BASE)
TEMA=core.elegir_intencion(BASE,"-".join([CODIGO_1.strip(),CODIGO_2.strip()]))
print("Corpus SECOP:",len(DOCUMENTOS),"IDs únicos · tu caso:",TEMA["titulo"],"· código:",TEMA["id"])
core.escribir(TRABAJO/"identidad.json",{"pareja":PAREJA,"integrantes":[{"nombre":INTEGRANTE_1,"codigo":CODIGO_1,"apellido":APELLIDO_1},{"nombre":INTEGRANTE_2,"codigo":CODIGO_2,"apellido":APELLIDO_2}] if CODIGO_2.strip() else [{"nombre":INTEGRANTE_1,"codigo":CODIGO_1,"apellido":APELLIDO_1}],"intencion":TEMA["id"]})
print("Tu pregunta literal:",TEMA["literal"],"\nTu paráfrasis:",TEMA["parafrasis"])

## Inicio · El error que cambia la investigación

Laura escribe dos consultas para encontrar procesos similares. Una utiliza palabras del objeto contractual, la otra una **paráfrasis** (mismo propósito escrito de otra manera).

**Predice antes de ejecutar:** ¿esperas que ambas produzcan el mismo Top-5?

**No se califican las predicciones.** Una predicción equivocada es material para aprender. Verás después qué cambió y por qué.

**Cómo se lee:** dos consultas describen la misma intención del equipo. **Qué nos dice:** los términos cambian. **Qué NO permite concluir todavía:** cuál de ellas recupera más contratos relevantes. **Qué error común:** confundir una frase parecida con un ranking idéntico.

In [ ]:
#@title Inicio · ELIGE · Predicción inicial (formativa) { display-mode: "form" }
PREDICCION_INICIAL = "— selecciona —" #@param ["— selecciona —","Obtendrán exactamente el mismo Top-5","Podrían cambiar los documentos y el orden","La segunda siempre será mejor"]
if PREDICCION_INICIAL=="— selecciona —":raise ValueError("Selecciona tu predicción antes de correr la búsqueda.")
core.escribir(TRABAJO/"prediccion.json",{"antes_de_buscar":PREDICCION_INICIAL,"literal":TEMA["literal"],"parafrasis":TEMA["parafrasis"]})
print("Predicción guardada, todavía no sabemos si es correcta.")

## E1 · ¿Por qué este contrato aparece primero? BM25 — 20 puntos

**El problema:** la búsqueda literal puede fallar incluso con una pregunta válida. Para entenderlo hay que mirar el *tokenizador* (divide el texto en unidades), la *frecuencia del término* (cuántas veces aparece) y la *rareza* (si aparece en pocos documentos). BM25 usa esas señales y también la longitud.

**Ejemplo manual:** tres documentos: A = «mantenimiento aeronaves», B = «mantenimiento equipos», C = «estadio deportivo». La consulta «mantenimiento aeronaves» coincide con A en dos términos y con B en uno. No es una prueba de relevancia semántica, pero explica la puntuación léxica.

**Qué vas a hacer:** inspeccionar tokens de TU consulta, calcular dos rankings, agregar una sola palabra y ver si cambia el resultado. Este BM25 se ejecuta **localmente en Python**; no lo presentamos como una consulta remota de Elasticsearch.

**Cómo se lee:** cada fila tiene un ID, un puntaje BM25 y un lugar. **Qué nos dice:** orden léxico para ese corpus. **Qué NO permite concluir todavía:** la pertinencia humana. **Qué error común:** comparar directamente un score BM25 con un coseno de E5.

In [ ]:
# E1.1 y E1.2 · EJECUTA · Analizar tokens y comparar dos Top-10
from IPython.display import display
import pandas as pd
TOKENS_LITERAL=core.tokens(TEMA["literal"])
RANK_BM_LITERAL=core.bm25(DOCUMENTOS,TEMA["literal"],10)
RANK_BM_PARAFRASIS=core.bm25(DOCUMENTOS,TEMA["parafrasis"],10)
print("Tokens de la consulta literal:",TOKENS_LITERAL)
print("Literal (Top-5):");display(pd.DataFrame(RANK_BM_LITERAL[:5]))
print("Paráfrasis (Top-5):");display(pd.DataFrame(RANK_BM_PARAFRASIS[:5]))
print("Coinciden posiciones:",[r["id_proceso"] for r in RANK_BM_LITERAL[:5]]==[r["id_proceso"] for r in RANK_BM_PARAFRASIS[:5]])

### E1.3 · Cambia una palabra, no todo el sistema

Selecciona una palabra y agrégala al final de la pregunta literal. Así podrás atribuir el cambio a una variable.

Si el ranking **no** cambia, ese también es un resultado válido: comprueba si el término estaba presente en la colección, si aportaba información nueva o si otros términos dominaron el score. No inventes un cambio para recibir puntos.

**Pregunta guía:** ¿agregar palabras siempre mejora la búsqueda? La respuesta depende de los documentos, no del botón.

In [ ]:
#@title E1.3 · ELIGE · Agrega UN término, predice y compara { display-mode: "form" }
PALABRA_NUEVA = "— selecciona —" #@param ["— selecciona —","adquisición","mantenimiento","servicios","construcción","instalaciones"]
PREDICCION_CAMBIO = "— selecciona —" #@param ["— selecciona —","Cambiará el primer resultado","Se mantendrá el primer resultado","No puedo saberlo sin ejecutar"]
if "—" in (PALABRA_NUEVA,PREDICCION_CAMBIO):raise ValueError("Elige el término y la predicción.")
CONSULTA_CAMBIADA=TEMA["literal"]+" "+PALABRA_NUEVA
RANK_CAMBIO=core.bm25(DOCUMENTOS,CONSULTA_CAMBIADA,10)
print("Antes:",[x["id_proceso"] for x in RANK_BM_LITERAL[:5]])
print("Después:",[x["id_proceso"] for x in RANK_CAMBIO[:5]])
print("Primera posición cambió:",RANK_CAMBIO[0]["id_proceso"]!=RANK_BM_LITERAL[0]["id_proceso"])
core.escribir(TRABAJO/"E1/bm25.json",{
 "analizador":"normalizado_simple","tokens_literal":TOKENS_LITERAL,
 "literal":RANK_BM_LITERAL,"parafrasis":RANK_BM_PARAFRASIS,
 "cambio":{"consulta":CONSULTA_CAMBIADA,"palabra":PALABRA_NUEVA,"prediccion":PREDICCION_CAMBIO,"ranking":RANK_CAMBIO}})
checkpoint("E1")

## E2 · ¿Cómo encontrar contratos con otras palabras? — 25 puntos

**Vector:** lista ordenada de números. **Embedding:** representación numérica que produce un modelo de un texto. **Similitud coseno:** compara direcciones de los vectores; cerca de 1 indica mayor semejanza geométrica, no necesariamente relevancia para una persona.

**Ejemplo manual:** A=(1,0), B=(0,1), consulta=(2,0). Coseno(A,consulta)=1 y coseno(B,consulta)=0, aunque la consulta tenga magnitud 2. En el taller E5 produce **384 valores**, no dos.

El modelo **intfloat/multilingual-e5-small** necesita prefijo `passage:` para documentos y `query:` para preguntas. Guardar los vectores permite comparar ambos mecanismos sobre exactamente los mismos 200 procesos.

**Qué se evalúa:** representación E5, Top-k por coseno, experiencia de base vectorial y un experimento con una variable. No se vuelve a calificar la carga documental convencional del TC1.

**Cómo se lee:** `(200,384)` son 200 vectores de 384 componentes. **Qué nos dice:** todos los documentos tienen representación. **Qué NO permite concluir todavía:** que se haya buscado en Atlas o que los vecinos sean relevantes. **Qué error común:** llamar «búsqueda remota» a una operación local de NumPy.

In [ ]:
#@title E2.1 y E2.2 · EJECUTA · E5 real y dos Top-10 por coseno { display-mode: "form" }
import numpy as np
from sentence_transformers import SentenceTransformer
MODELO_E5=SentenceTransformer(core.MODELO)
MODELO_E5.max_seq_length=256
CARPETA_E2=TRABAJO/"E2";CARPETA_E2.mkdir(parents=True,exist_ok=True)
F_EMB=CARPETA_E2/"embeddings.npy"
if F_EMB.exists():
    EMB=np.load(F_EMB,allow_pickle=False)
    if EMB.shape!=(len(DOCUMENTOS),384):raise ValueError("Caché de dimensión incorrecta: borra embeddings.npy.")
else:
    EMB=np.asarray(MODELO_E5.encode(["passage: "+core.texto(d) for d in DOCUMENTOS],
                batch_size=16,normalize_embeddings=True,show_progress_bar=True),dtype=np.float32)
    np.save(F_EMB,EMB)
V_LITERAL=np.asarray(MODELO_E5.encode(["query: "+TEMA["literal"]],normalize_embeddings=True)[0],dtype=np.float32)
V_PARAFRASIS=np.asarray(MODELO_E5.encode(["query: "+TEMA["parafrasis"]],normalize_embeddings=True)[0],dtype=np.float32)
assert EMB.shape==(len(DOCUMENTOS),384) and np.isfinite(EMB).all()
np.save(CARPETA_E2/"query_literal.npy",V_LITERAL)
np.save(CARPETA_E2/"query_parafrasis.npy",V_PARAFRASIS)
RANK_SEM_LITERAL=core.coseno(DOCUMENTOS,EMB,V_LITERAL,10)
RANK_SEM_PARAFRASIS=core.coseno(DOCUMENTOS,EMB,V_PARAFRASIS,10)
print("Modelo real:",core.MODELO,"· dimensiones:",EMB.shape)
print("Top-5 semántico (NumPy local, NO Atlas):")
display(pd.DataFrame(RANK_SEM_PARAFRASIS[:5]))

### E2.3 · Base vectorial: ¿dónde se ejecutó la consulta?

**MongoDB Atlas Vector Search** permite persistir vectores y consultarlos en un motor remoto. Su índice vectorial no es el índice convencional que se evaluó en el TC1. La consulta usa `$vectorSearch`, `path`, `queryVector`, `limit` y opcionalmente `numCandidates`.

**Separa tres pruebas:** un `ping` demuestra conexión; un índice `queryable` demuestra disponibilidad; solo un resultado devuelto por `aggregate` demuestra consulta remota.

La ruta recomendada intenta **Atlas temporal Free** con credenciales individuales que jamás se escriben en el ZIP. Si el proveedor no responde, usa la ruta local declarada y continúa; el docente revisará el estado de la contingencia. No confundas una prueba local con Atlas.

**Privacidad:** nunca muestres la cadena de conexión, contraseña ni el enlace de reclamación dentro de una evidencia.

In [ ]:
#@title E2.3 · ELIGE · Intentar Atlas real o dejar ruta local claramente identificada { display-mode: "form" }
MODO_BASE_VECTORIAL = "Intentar Atlas temporal" #@param ["Intentar Atlas temporal","Ruta local (NO es Atlas)"]
import time,requests,secrets
from pymongo import MongoClient,UpdateOne
from pymongo.operations import SearchIndexModel
ATLAS={"motor":"NO EJECUTADO","sistema":"Ruta local declarada","ping":False,
       "indice_queryable":False,"query_ejecutada":False,"resultados":[],
       "indice_path":"embedding","indice_dimensiones":384,"pipeline_limit":5}
if MODO_BASE_VECTORIAL=="Intentar Atlas temporal":
    try:
        resp=requests.post("https://cloud.mongodb.com/api/atlas/v2/unauth/ephemeralClusters:create",
                headers={"Accept":"application/vnd.atlas.preview+json","Content-Type":"application/json"},
                json={"clusterName":"TC2-"+secrets.token_hex(4)},timeout=50)
        resp.raise_for_status();datos_cluster=resp.json()
        uri=datos_cluster["connectionString"]
        cliente=MongoClient(uri,serverSelectionTimeoutMS=12000)
        cliente.admin.command("ping");ATLAS["ping"]=True
        col=cliente["curso_tc2"]["procesos_vector"]
        lote=hashlib.sha256(("TC2-"+PAREJA+TEMA["id"]).encode()).hexdigest()[:16]
        col.bulk_write([UpdateOne({"_id":d["id_proceso"],"lote":lote},
                  {"$set":{"id_proceso":d["id_proceso"],"lote":lote,
                           "nombre_proceso":d["nombre_proceso"],"embedding":v.tolist()}},upsert=True)
                   for d,v in zip(DOCUMENTOS,EMB)],ordered=False)
        nombre_indice="tc2_idx_"+lote[:8]
        definicion={"fields":[{"type":"vector","path":"embedding","numDimensions":384,"similarity":"cosine"},
                               {"type":"filter","path":"lote"}]}
        try:col.create_search_index(model=SearchIndexModel(definition=definicion,name=nombre_indice,type="vectorSearch"))
        except Exception as error:
            if "already exists" not in str(error).lower():raise
        for intento in range(15):
            indices=list(col.list_search_indexes())
            ATLAS["indice_queryable"]=any(i.get("name")==nombre_indice and i.get("queryable") for i in indices)
            if ATLAS["indice_queryable"]:break
            time.sleep(4)
        if ATLAS["indice_queryable"]:
            consulta=[{"$vectorSearch":{"index":nombre_indice,"path":"embedding","queryVector":V_PARAFRASIS.astype(float).tolist(),
                        "numCandidates":100,"limit":5,"filter":{"lote":lote}}},
                      {"$project":{"_id":0,"id_proceso":1,"nombre_proceso":1,"score":{"$meta":"vectorSearchScore"}}}]
            encontrados=list(col.aggregate(consulta,maxTimeMS=30000))
            ATLAS.update({"sistema":"Atlas real","motor":"MongoDB Atlas Vector Search","query_ejecutada":True,
                "resultados":encontrados})
            print("ATLAS REAL:",len(encontrados),"resultados, índice queryable.")
        else: print("ATLAS: índice no disponible; NO se marca consulta remota.")
        # Nunca guardamos uri ni claimUrl en los archivos del taller
        del uri,cliente,datos_cluster
    except Exception as error:
        ATLAS["error_tipo"]=type(error).__name__
        print("ATLAS NO EJECUTADO:",type(error).__name__,"· continúa local, sin fingir resultados.")
else:
    print("NUMPY LOCAL, Atlas NO EJECUTADO por decisión del equipo.")
print("Estado Atlas:",{k:v for k,v in ATLAS.items() if k not in ("resultados",)})

### E2.4 · Cambia el número de resultados

**Top-k:** los k vecinos de puntuación más alta. Cambiar k de 5 a 3 u 8 cambia cuántos documentos recuperas, pero **no** enseña que el índice sea mejor.

Predice y elige solo el nuevo k; el cuaderno recalculará sobre los mismos embeddings. Si en Atlas no se ejecutó, se explicitará el uso de NumPy local.

In [ ]:
#@title E2.4 · ELIGE · Un Top-k alternativo { display-mode: "form" }
TOP_K_ALTERNO = 3 #@param [3,8] {type:"raw"}
RANK_ALTERNO=core.coseno(DOCUMENTOS,EMB,V_PARAFRASIS,int(TOP_K_ALTERNO))
print("Antes Top-5:",[x["id_proceso"] for x in RANK_SEM_PARAFRASIS[:5]])
print("Ahora Top-",TOP_K_ALTERNO,":",[x["id_proceso"] for x in RANK_ALTERNO])
core.escribir(TRABAJO/"E2/semantica.json",{"modelo":core.MODELO,"prefijos":["passage:","query:"],
 "literal":RANK_SEM_LITERAL,"parafrasis":RANK_SEM_PARAFRASIS,
 "atlas":ATLAS,"experimento":{"k":int(TOP_K_ALTERNO),"ranking":RANK_ALTERNO}})
checkpoint("E2")

## E3 · ¿Cuál buscador recupera documentos útiles? — 15 puntos

El score del motor NO es una calificación humana. La **pertinencia** se evalúa comparando el ranking con un conjunto de documentos relevantes identificado de forma independiente para tu intención.

**Precision@5:** relevantes recuperados entre cinco resultados. **Recall@5:** relevantes recuperados dividido por todos los relevantes conocidos para esa intención en nuestro corpus.

**Ejemplo manual:** de cinco resultados, tres son relevantes; en el corpus se han etiquetado seis relevantes. Precision@5=3/5=0,6 y Recall@5=3/6=0,5. No lo confundas con solapamiento ANN–ENN.

**RRF (Reciprocal Rank Fusion):** combina lugares de dos rankings sin comparar el score de BM25 con el coseno. Cada posición r aporta `1/(60+r)` y se suman los aportes.

**Tu reto:** calcular las métricas y elegir BM25, E5 o RRF a partir de TU resultado, descartando una alternativa y señalando qué evidencia falta.

In [ ]:
# E3.1 y E3.2 · EJECUTA · Relevancia y fusión RRF
RELEVANTES=TEMA["relevantes"]
METRICAS_BM=core.medidas(RANK_BM_PARAFRASIS,RELEVANTES)
METRICAS_E5=core.medidas(RANK_SEM_PARAFRASIS,RELEVANTES)
RANK_RRF=core.fusion_rrf(RANK_BM_PARAFRASIS,RANK_SEM_PARAFRASIS,60,10)
METRICAS_RRF=core.medidas(RANK_RRF,RELEVANTES)
print("Juicios de relevancia del curso:",len(RELEVANTES),"· intención:",TEMA["titulo"])
display(pd.DataFrame([dict(motor="BM25",**METRICAS_BM),dict(motor="E5",**METRICAS_E5),dict(motor="RRF",**METRICAS_RRF)]))
print("Ranking combinado:");display(pd.DataFrame(RANK_RRF[:5]))
print("Qué NO se puede concluir:",TEMA["limite"])

In [ ]:
#@title E3.3 · ELIGE · Motor, alternativa y límite de tu decisión { display-mode: "form" }
MOTOR_ELEGIDO = "— selecciona —" #@param ["— selecciona —","BM25","E5","RRF"]
MOTOR_DESCARTADO = "— selecciona —" #@param ["— selecciona —","BM25","E5","RRF"]
LIMITE_RANKING = "— selecciona —" #@param ["— selecciona —","Solo cubre los juicios etiquetados","No demuestra irregularidades","El mayor score prueba fraude"]
if MOTOR_ELEGIDO=="— selecciona —" or MOTOR_DESCARTADO=="— selecciona —" or LIMITE_RANKING=="— selecciona —":
    raise ValueError("Elige tres respuestas; no hay respuesta preseleccionada.")
if MOTOR_ELEGIDO==MOTOR_DESCARTADO:raise ValueError("Debes descartar una alternativa distinta de la elegida.")
metricas={"BM25":METRICAS_BM,"E5":METRICAS_E5,"RRF":METRICAS_RRF}
EVAL={"bm25":METRICAS_BM,"e5":METRICAS_E5,"rrf":RANK_RRF,"metricas_rrf":METRICAS_RRF,
 "decision":{"elegido":MOTOR_ELEGIDO,"descartado":MOTOR_DESCARTADO,
             "limite":LIMITE_RANKING,"precision_observada":metricas[MOTOR_ELEGIDO]["precision_at_5"]}}
core.escribir(TRABAJO/"E3/evaluacion.json",EVAL)
print("Elegiste",MOTOR_ELEGIDO,"con Precision@5",metricas[MOTOR_ELEGIDO]["precision_at_5"])
print("Alternativa descartada:",MOTOR_DESCARTADO)
checkpoint("E3")

## E4 · ¿Qué sucede con una imagen cuando la reducimos? — 20 puntos

Para evitar fotografías sin licencia, generarás una **imagen real desde el ranking RRF de tu equipo**. Es un gráfico, no una evidencia fotográfica del proceso contractual.

**Píxel:** posición de una imagen con tres canales RGB. **Luminancia:** aproximación a su claridad, `0.2126 R + 0.7152 G + 0.0722 B`. **Histograma:** conteo de píxeles por intervalos de luminancia.

**Ejemplo manual:** RGB (255,0,0) tiene luminancia aproximada 54.21. No identifica objetos ni palabras. Una imagen de 960×520 píxeles contiene 499.200 posiciones; la suma de todos los bins debe coincidir con ese número.

**Decisión profesional:** dos tamaños pueden comprimir el mismo gráfico, pero si uno hace ilegible el ID del proceso, es inútil para una revisión verificable. Comprobarás dos perfiles: 160 y 640 píxeles en el lado mayor.

**Cómo se lee:** ejes del histograma = brillo y conteos. **Qué nos dice:** distribución tonal. **Qué NO permite concluir todavía:** qué contrato es irregular o qué texto lee un OCR. **Qué error común:** concluir que un histograma nítido implica texto legible.

In [ ]:
# E4.1 y E4.2 · EJECUTA · Imagen derivada del ranking y dos perfiles
from IPython.display import Image as MostrarImagen
CARPETA_E4=TRABAJO/"E4";CARPETA_E4.mkdir(exist_ok=True)
IMG=core.csv_image(RANK_RRF,DOCUMENTOS,CARPETA_E4/"ranking.png")
INSPECCION=core.inspeccion_imagen(IMG)
PERFIL_A=core.perfil_imagen(IMG,CARPETA_E4/"perfil_160.png",160,0.85)
PERFIL_B=core.perfil_imagen(IMG,CARPETA_E4/"perfil_640.png",640,1.10)
print("Píxel RGB:",INSPECCION["pixel_rgb"],"· luminancia:",INSPECCION["pixel_luminancia"])
print("Píxeles vs suma histograma:",INSPECCION["ancho"]*INSPECCION["alto"],INSPECCION["suma_histograma"])
print("Perfil 160:",PERFIL_A["ancho"],"×",PERFIL_A["alto"]," · bytes:",PERFIL_A["bytes"])
print("Perfil 640:",PERFIL_B["ancho"],"×",PERFIL_B["alto"]," · bytes:",PERFIL_B["bytes"])
display(MostrarImagen(filename=str(CARPETA_E4/"perfil_160.png")))
display(MostrarImagen(filename=str(CARPETA_E4/"perfil_640.png")))

### E4.3 · La mejor transformación depende de la tarea

Mira las dos versiones de tu gráfica. ¿Puedes leer el identificador y la explicación de los resultados en el tamaño menor?

No hay que escoger el archivo más pequeño por costumbre: el objetivo es conservar la **información necesaria para interpretar el resultado**.

Al comparar histograms, no confundas reducción de dimensiones con cambios en el número de bins. El número de píxeles sí cambia al reducir la imagen; cambiar solo la cantidad de bins no lo cambia.

In [ ]:
#@title E4.3 · ELIGE · Perfil para un analista que necesita leer etiquetas { display-mode: "form" }
PERFIL_PREFERIDO = "— selecciona —" #@param ["— selecciona —","160","640"]
LIMITE_IMAGEN = "— selecciona —" #@param ["— selecciona —","Reducir puede volver ilegibles las etiquetas","Todo archivo pequeño conserva el mismo detalle","El histograma demuestra el contenido semántico"]
if "—" in (PERFIL_PREFERIDO,LIMITE_IMAGEN):raise ValueError("Selecciona perfil y límite.")
core.escribir(CARPETA_E4/"imagen.json",{"inspeccion":INSPECCION,
 "perfiles":{"160":PERFIL_A,"640":PERFIL_B},
 "preferido":PERFIL_PREFERIDO,"limite":LIMITE_IMAGEN})
checkpoint("E4")

## E5 · ¿Qué nos dice realmente una voz? — 15 puntos

**Diez grabaciones auténticas de voz en español:** PolyAI / MINDS14 es-ES, **CC BY 4.0**. Cada una incluye audio real, una intención declarada en el dataset y una transcripción humana de referencia. **No son audios de noticias ni están vinculadas a SECOP.**

**Muestra:** un valor de la onda en un instante. **Frecuencia de muestreo (Hz):** muestras por segundo, no el tono de la voz. **RMS:** raíz del promedio de cuadrados de la amplitud; para [1,-1,0,0], RMS ≈ 0.707. **FFT:** representa frecuencias presentes en una ventana. **Espectrograma:** evolución de la energía por frecuencias a lo largo del tiempo.

**Reto:** escuchar un archivo, medir el lote de diez y cambiar de una ventana FFT de 512 a 1024 muestras sin cambiar el audio. Comprueba cómo disminuye la separación de bins de frecuencia `sr/n`.

**Cómo se lee:** RMS resume energía de amplitud; FFT muestra componentes de frecuencia. **Qué nos dice:** propiedades físicas de la señal. **Qué NO permite concluir todavía:** palabras, emoción o intención del hablante. **Qué error común:** declarar que se ejecutó ASR cuando solo se leyó la referencia del dataset.

**Si falla descarga:** el cuaderno se detiene con el error real. Reintenta la conexión; no se fabricarán diez voces sintéticas para superar el control.

In [ ]:
#@title E5.1 · EJECUTA · Obtener diez WAV humanos autorizados (red real, ~39 MB) { display-mode: "form" }
CARPETA_E5=TRABAJO/"E5";CARPETA_E5.mkdir(exist_ok=True)
AUDIOS=core.obtener_voces(CARPETA_E5/"audio")
print("Voces reales:",len(AUDIOS),"· licencias:",set(x["license"] for x in AUDIOS))
assert len(AUDIOS)==10 and len({a["source_record_id"] for a in AUDIOS})==10
display(pd.DataFrame([{"voz":a["n"],"rms":a["medidas"]["rms"],
    "segundos":a["medidas"]["duracion_s"],"Hz":a["medidas"]["sr"],
    "intencion_dataset":a["intent_class"]} for a in AUDIOS]))
print("Las transcripciones proceden del dataset, NO de un modelo ASR ejecutado por TC2.")

In [ ]:
#@title E5.2 · ELIGE · Reproducir voz y comparar dos ventanas FFT { display-mode: "form" }
POSICION_VOZ = 1 #@param {type:"integer"}
if not 1<=int(POSICION_VOZ)<=10:raise ValueError("Selecciona una posición 1–10 de TU tabla.")
from IPython.display import Audio,display
RUTA_VOZ=CARPETA_E5/"audio"/AUDIOS[int(POSICION_VOZ)-1]["archivo"]
METRICA_512=core.metricas_voz(RUTA_VOZ,512)
METRICA_1024=core.metricas_voz(RUTA_VOZ,1024)
print("Ventana 512:",METRICA_512)
print("Ventana 1024:",METRICA_1024)
print("Transcripción humana DE REFERENCIA:",AUDIOS[int(POSICION_VOZ)-1]["transcripcion_referencia"])
display(Audio(filename=str(RUTA_VOZ)))
import matplotlib.pyplot as plt
muestras,sr=core.audio_pcm(RUTA_VOZ)
fig,ax=plt.subplots(2,1,figsize=(11,6),layout="constrained")
ax[0].plot(np.arange(min(len(muestras),sr))/sr,muestras[:sr],linewidth=.6)
ax[0].set(xlabel="Tiempo (s)",ylabel="Amplitud",title="Forma de onda (primer segundo)")
ax[1].specgram(muestras[:min(len(muestras),sr*4)],Fs=sr,NFFT=512,noverlap=256)
ax[1].set(xlabel="Tiempo (s)",ylabel="Frecuencia (Hz)",title="Espectrograma · ventana 512")
plt.show()

In [ ]:
#@title E5.3 · ELIGE · Qué NO demuestra un espectro y conservar evidencia { display-mode: "form" }
LIMITE_AUDIO = "— selecciona —" #@param ["— selecciona —","RMS y FFT no transcriben palabras","RMS demuestra la intención del hablante","Un espectrograma identifica fraudes"]
REFERENCIA_HUMANA_NO_ES_ASR = False #@param {type:"boolean"}
if LIMITE_AUDIO=="— selecciona —":raise ValueError("Elige el límite antes de guardar.")
comparacion={"posicion":int(POSICION_VOZ),"ventana_512":METRICA_512,"ventana_1024":METRICA_1024,
             "limite":LIMITE_AUDIO,"referencia_no_asr":REFERENCIA_HUMANA_NO_ES_ASR}
core.escribir(CARPETA_E5/"audios.json",{"audios":AUDIOS,"comparacion":comparacion})
print("Se conservan las diez fuentes y el experimento. No hay asociación con SECOP.")
checkpoint("E5")

## E6 · La demostración de Laura — 5 puntos

Tu producto final no consiste en sumar pantallazos. El notebook genera una página **HTML de resultados ya calculados**, con la comparación de BM25, E5 y RRF, el gráfico derivado de TU ranking y las diez voces reales con metadatos de procedencia.

En la página, filtrar la tabla o abrir un audio **no ejecuta una nueva inferencia semántica**. Las consultas ya se realizaron en Colab. Los WAV de PolyAI forman una colección aparte: el sistema permite analizarlos, no asociarlos falsamente con contratos.

**Para defender:** elige qué mecanismo usarías y qué NO debes inferir de píxeles y espectros. Las respuestas son estructuradas, no ensayos que pueda redactar un asistente.

In [ ]:
#@title E6.1 y E6.2 · ELIGE · Defensa del producto final { display-mode: "form" }
MODELO_DEFENSA = "— selecciona —" #@param ["— selecciona —","BM25","E5","RRF"]
QUE_NO_DICE_AUDIO = "— selecciona —" #@param ["— selecciona —","Transcripción ni intención","Nada, el espectrograma lo revela todo"]
QUE_NO_DICE_IMAGEN = "— selecciona —" #@param ["— selecciona —","Contenido semántico del contrato","Nada, el histograma muestra la relevancia"]
if "— selecciona —" in (MODELO_DEFENSA,QUE_NO_DICE_AUDIO,QUE_NO_DICE_IMAGEN):raise ValueError("Completa las tres decisiones.")
DEFENSA={"modelo":MODELO_DEFENSA,"audio_no_implica":QUE_NO_DICE_AUDIO,"imagen_no_implica":QUE_NO_DICE_IMAGEN,
         "intencion":TEMA["id"],"imagen_perfil":PERFIL_PREFERIDO,"audio_posicion":int(POSICION_VOZ)}
core.escribir(TRABAJO/"E6/defensa.json",DEFENSA)
PAGINA=core.hacer_html(TRABAJO,DOCUMENTOS,RANK_BM_PARAFRASIS,RANK_SEM_PARAFRASIS,RANK_RRF,INSPECCION,AUDIOS,DEFENSA)
print("Producto generado:",PAGINA,"· se puede abrir tras extraer el ZIP; los WAV se enlazan por rutas relativas.")
checkpoint("E6")

## Autoevaluación antes de entregar

El **validador lee los archivos**, no las variables en memoria. Comprueba operaciones BM25/coseno, juicios de relevancia, RRF, píxeles, archivos de audio y el explorador.

**Si algo aparece en ✗:** vuelve al desafío correspondiente, corrige y revalida. La pérdida de una etapa no anula las siguientes: puedes continuar con los resultados de referencia para estudiar, pero no se considera que hayas ejecutado la etapa fallida.

**Advertencia:** E2.3 (Atlas) exige revisión docente del registro y de contingencias; el autoevaluador no puede certificar por sí solo una llamada remota. Los puntos son provisionales hasta la revisión de evidencia.

**Escala como TC1:** 0–100, con fórmula propuesta 1 + 4 × puntos/100 (0,0 sin entrega). El docente confirma la ponderación oficial vigente.

In [ ]:
#@title COMPRUEBA · Validación independiente y retroalimentación por control { display-mode: "form" }
INFORME=core.validar(TRABAJO,DOCUMENTOS,TEMA)
for cod,detalle in INFORME["controles"].items():
    print(("✓ " if detalle["ok"] else "✗ ")+cod+": "+str(detalle["puntos"])+"/"+str(detalle["maximo"])+" · "+detalle["detalle"])
print("PUNTAJE TÉCNICO PROVISIONAL:",INFORME["puntaje"],"/100")
print("REVISIÓN DOCENTE:",INFORME["pendiente_revision_docente"])
core.escribir(TRABAJO/"manifest_tc2.json",{"version":"TC2 V1","pareja":PAREJA,
 "intencion":TEMA["id"],"reporte":INFORME,"fecha_utc":datetime.datetime.now(datetime.timezone.utc).isoformat()})

In [ ]:
#@title ENTREGAR · Generar ZIP y manifest externo; verificar huella { display-mode: "form" }
ZIP=core.empaquetar(TRABAJO,INFORME)
MANIFEST_EXTERNO=TRABAJO.parent/"manifest_tc2.json"
shutil.copyfile(TRABAJO/"manifest_tc2.json",MANIFEST_EXTERNO)
HUELLA=core.sha(MANIFEST_EXTERNO)
print("ZIP:",ZIP,"\nMANIFEST:",MANIFEST_EXTERNO,"\nSHA-256 DEL MANIFEST:",HUELLA)
print("Nombre del notebook ejecutado: TC2_"+PAREJA+".ipynb")
print("ASUNTO DEL CORREO: [BIG DATA 2026-2S][TC2] "+PAREJA.upper()+" - "+APELLIDO_1.upper()+(" - "+APELLIDO_2.upper() if APELLIDO_2 else ""))
print("Si estás en Colab, Archivo → Descargar → .ipynb; renómbralo TC2_"+PAREJA+".ipynb")
try:
    from google.colab import files
    print("Descargando el ZIP y el manifest; si se bloquean ventanas emergentes, descárgalos desde el panel Archivos.")
    files.download(str(ZIP))
    files.download(str(MANIFEST_EXTERNO))
except ImportError:
    print("Se ejecutó fuera de Colab: localiza los dos archivos en la ruta impresa.")

## Entrega TC2 · Google Drive restringido + correo institucional

**Bajo las mismas condiciones del TC1**, una pareja entrega una sola carpeta restringida de Google Drive, compartida con **jzaineam@ucentral.edu.co** en permiso **Lector**, y un único correo institucional. El LMS no registra automáticamente tu nota TC2 mientras no exista integración verificada.

En la carpeta deben existir exactamente estos **tres archivos**:

```text
TC2_BIGDATA_<PAREJA_ID>_<APELLIDOS>/
├── TC2_<PAREJA_ID>.ipynb     ← descargado desde Colab con las salidas ejecutadas
├── TC2_<PAREJA_ID>.zip       ← incluye E1–E6, explorador y audios
└── manifest_tc2.json        ← autoevaluación por control y huella verificable
```

**Correo:** un integrante escribe a **jzaineam@ucentral.edu.co**, copia al compañero, usa el asunto que imprime «ENTREGAR» y pone el enlace restringido, integrantes y **SHA-256 del manifest**. Si corriges la entrega, reemplaza archivos y responde en el mismo hilo con la huella nueva. Nunca compartas públicamente el ZIP, porque contiene identificaciones del equipo.

**Fecha y hora máxima:** el docente las anunciará por LMS y correo; todavía no están fijadas en esta versión de preparación. No supongas que la fecha del TC1 sirve para TC2.

**Importante:** el último paso genera ZIP y manifest. Tú descargas el **notebook ejecutado** usando Archivo → Descargar → .ipynb, porque Colab no permite al paquete exportar de forma segura y automática las salidas de cualquier copia sin conocer su ubicación. No envíes una copia sin resultados.

**No se requieren capturas obligatorias de Atlas o cuentas externas**, porque los puntos nuevos se sustentan en archivos recalculables. Si Atlas se ejecuta, el docente puede pedir evidencia adicional para corroborar la conexión real.

## Cierre · Lo que Laura ya sabe y lo que todavía no

| Problema | Resultado del taller | Lo que no demuestra |
|---|---|---|
| No encuentro documentos con palabras distintas | BM25 y E5, comparados con preguntas y IDs propios | Relevancia universal |
| ¿Cómo priorizo el top de la consulta? | Precision@5, Recall@5 y RRF sobre juicios curados | Fraude ni irregularidad contractual |
| ¿Qué sucede al reducir una imagen? | Histogramas, píxel RGB y comparación de perfiles | Comprensión semántica de la foto |
| ¿Qué dice una voz? | Forma de onda, RMS, FFT y espectrograma de diez WAV reales | Palabras ni emociones sin otros métodos |
| ¿Puedo integrar herramientas? | Explorador offline con dos inventarios separados | Asociación entre contratos y voces sin clave común |

**Idea para llevar:** un motor recupera candidatos, un juicio de relevancia evalúa su utilidad y un análisis de señal caracteriza archivos; son mecanismos distintos. Un sistema confiable debe mostrar su procedencia, mediciones y límites.

### Hoja de trucos
- BM25 → tokens, rareza, frecuencia, longitud; no igual a coseno.
- E5 → `passage:` (documentos), `query:` (pregunta), vector de 384 componentes.
- Coseno → proximidad geométrica, no relevancia humanamente etiquetada.
- Precision@5 → relevantes encontrados/5.
- Recall@5 → relevantes encontrados/relevantes del universo etiquetado.
- RRF → suma de `1/(60+rango)` por mecanismo.
- Luminancia → `0.2126R+0.7152G+0.0722B`.
- FFT → la resolución nominal aproximada es frecuencia de muestreo/longitud de ventana.
- RMS y espectrograma → propiedades físicas, no transcripción.
- `core.validar(...)` → lectura independiente de los archivos.
- Si Colab se reinicia: vuelve a Preparar e Identidad, restaura avance Drive si lo activaste, y reejecuta las etapas de las que dependan tus variables.